> **실험 기록.** 속도 모델 설계 과정의 비교 실험이며 출력은 실행 당시 결과를 보존했다. 최종 모델은 `src/models/`, 공식 성능은 [docs/SPEED_MODEL.md](../docs/SPEED_MODEL.md), 노트북별 요약과 재실행 조건은 [notebooks/README.md](README.md)에 있다.

# Speed Prediction — 파생/추가 피처 빠른 테스트 (LightGBM)

**목적**: `01_speed_model_baseline.ipynb`의 LightGBM 결과를 기준점으로 두고, 파생 피처와 신규 외부 피처를 추가했을 때 실제로 개선되는지 확인합니다.

**핵심 전제**: 여기서 만드는 피처들은 전부 **미관측 링크(29.25%)에도 쓸 수 있는 범용 피처** 후보를 찾는 작업입니다 — 관측 링크는 자기 자신의 과거 속도 이력(Model A 방식)을 쓰면 되므로 이 노트북의 범위가 아닙니다.

**평가 방식 변경**: 지금까지 매 버전마다 무작위 분할로 먼저 보고 마지막에 GroupKFold로 검산하는 2단계였는데, **무작위 분할을 뺐습니다.** 반복해서 확인한 결과 무작위 분할은 매번 GroupKFold보다 R²가 0.07~0.09 높게 나왔습니다(6차 랜덤 0.746 vs GroupKFold 0.680, 7차 랜덤 0.780 vs GroupKFold 0.708) — 같은 링크의 다른 시간대 행이 대부분 train에 있어서 이웃속도 피처가 거의 항상(93~97%) 살아있는, 우리가 실제로 풀어야 하는 "한 번도 안 본 도로" 문제보다 훨씬 쉬운 문제였기 때문입니다. `src/11` 공식 GroupKFold(0.562)가 노트북11의 무작위분할(0.610)보다 낮았던 첫 사례부터 이미 같은 신호였습니다. 이제부터는 **`LINK_ID` 기준 GroupKFold(공간 일반화) 하나만** 정식 평가로 씁니다 — 계산량도 절반 이상 줄어듭니다.

**버전 변경 이력**
- 1~4차, src/11 공식 평가: `data/README.md`·이전 커밋 참고(무작위분할 시절 기록)
- **5~7차**: 데이터셋 전환(dimension+fact) → 생활인구 추가 → 도로망 위상·강수파생 추가. GroupKFold 기준 R²: 0.680(7차, 단일모델).
- **8차(B/C 분리모델)**: `notebooks/04_tier_split_evaluation.ipynb`와 `src/models/02_build_model_bc.py`가 실전 적용한 방식. 이웃속도 피처가 있으면(B 계층) 도움되지만 없으면(C 계층) 해가 되는 문제를, 모델을 둘로 나눠(이웃속도 포함 모델 / 정적 전용 모델) 해결합니다. GroupKFold R²=0.709.
- **9차(View-T 실측 데이터)**: `AverageSpeed(LINK).xlsx`(2024년 링크별 실측 평균속도, 연간+시간대별), `CongestIndex(LINK).xlsx`(혼잡빈도·시간·기대강도)를 추가합니다. Level5.5(주요도로망) 17,929개 링크만 커버(C 계층 1.1%)하지만, `annual_avg_speed`는 우리 실측 속도와 상관계수 0.787로 최강 단일 피처 후보였습니다. GroupKFold R²=0.728(+0.019).
- **10차(2홉 이웃 확장, 블렌딩)**: 1홉 이웃이 없는 C 계층(전체 test의 5.5%) 중 91.1%가 2홉까지 넓히면 관측 이웃을 찾는다는 걸 그래프 구조로 사전 확인했습니다. 1홉(가중치 1.0)+2홉전용(가중치 0.5) 가중평균으로 9차의 이웃속도 피처를 교체했습니다. **결과**: C 계층 커버리지는 극적으로 개선(283,255행→24,560행, -91.3%)됐지만, 1홉이 이미 있던 행에도 2홉을 섞어 넣는 바람에 **B 계층 R²가 오히려 하락**(0.741→0.731)했습니다. 전체 R²는 거의 제자리(0.728→0.730).
- **11차(1홉 우선 + 2홉 폴백)**: 10차의 문제를 고치기 위해 블렌딩 대신 **폴백**으로 바꿉니다 — 1홉 이웃이 있으면 1홉만 쓰고(B 계층 안 건드림), 1홉이 없을 때만 순수 2홉 평균으로 대체합니다. B 계층 손실 없이 C 계층 커버리지 이득만 가져갈 수 있는지 확인합니다. 프로덕션(`src/models/02_build_model_bc.py`) 반영을 염두에 둔 설계입니다.
- **12차(코리도/도심거리 + 위상·고도 중복제거)**: `experiments/build_structural_features.py`의 도로명 그룹 코리도 규모(`rn_links`,`rn_len`,`rn_lanes`,`rn_maxspd`)와 서울시청 기준 도심거리(`dist_cbd`)를 추가하고, 그룹별 상관분석으로 확인된 중복 위상·고도 컬럼 6+5개를 제거했습니다. B/C 계층 동시 개선(R²=0.737, 이번 세션 최대 단일 개선폭).
- **13차(gain=0 13개 제거)**: 12차에서 5차례 연속 gain이 0에 가까웠던 강수파생 8개 + 경사방향/이상치 플래그 4개 + `congest_expect` 1개를 제거. 성능 손실 없이(오히려 소폭 개선) 피처 수만 줄임.
- **14차(추가정리 9개 + log_precip 복원 + 회전제한/교차로 4개, 미채택)**: 13차에서 9개 더 제거하고 `log_precip` 1개만 복원 + `MOCT_NODE.shp`/`TURNINFO.dbf` 기반 회전제한/교차로유형 4개(`turn_n_f/t`, `is_cross_f/t`)를 추가했으나, `is_cross_t`(gain 1.73%)는 유효했던 반면 추가 제거분이 손실을 내 전체 R²는 소폭 하락(0.737→0.7366). 13차를 기준선으로 유지하고 `is_cross_t`만 이후 버전에 반영하기로 함.
- **15차(13차와 동일 피처, LightGBM→XGBoost)**: 피처는 그대로 두고 알고리즘만 교체. R²=0.740으로 이번 세션 최고 기록 — 특히 C 계층(진짜 미관측) 분리모델이 0.288→0.315로 크게 개선됨. `notebooks/11` 베이스라인의 XGBoost 우위가 GroupKFold+B/C분리 조건에서도 재현됨을 확인하고, 이후 버전은 XGBoost를 기본으로 함.
- **16차(이번, 실행시간 절감을 위해 검증된 개선을 한 번에 묶음)**: 13차 피처 + `is_cross_t`(14차에서 유일하게 검증된 순수 이득) + `traffic_volume_ratio`(fact 테이블에 있었지만 그동안 안 쓰던 시간대별 교통량/AADT 비율) + `sample_count_5min`을 `sample_weight`로 반영(5분 표본 수가 많을수록 평균속도 라벨의 신뢰도가 높다고 보고 학습 시 가중치를 줌). 알고리즘은 15차에서 확인된 XGBoost.

**캐싱**: 원본 결합 결과를 `data/interim/notebook_cache/nb12_feature_cache_v8.parquet`에 캐싱합니다(16차에서 fact usecols에 `traffic_volume_ratio`·`sample_count_5min`을 추가하면서 캐시 스키마가 바뀌어 v7→v8로 올렸습니다).

In [1]:
import gc
from pathlib import Path

import geopandas as gpd
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DOCS_FIGURES_DIR = PROJECT_ROOT / "docs" / "figures" / "model"
DOCS_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# 새 스키마: 정적(링크당 1행) + 동적(링크x날짜x시간) 분리, LINK_ID로 merge
DIMENSION_PATH = PROCESSED_DIR / "seoul_link_dimension.csv"
FACT_PATH = PROCESSED_DIR / "seoul_link_speed_fact.csv"
SEOUL_LINK_SHP = INTERIM_DIR / "seoul_link.shp"

CACHE_PATH = INTERIM_DIR / "notebook_cache" / "nb12_feature_cache_v9.parquet"
CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

RANDOM_STATE = 42
COLOR_RMSE = "#2a78d6"   # dataviz 팔레트 slot 1 blue
COLOR_MAE = "#eb6834"    # dataviz 팔레트 slot 2 orange
COLOR_SEQ = "#3987e5"


## 1. 데이터 로드 (dimension + fact 결합, + 캐싱)

`seoul_link_speed_fact.csv`(동적, 15,094,335행)에 `seoul_link_dimension.csv`(정적, 63,548행)를 `LINK_ID`로 결합합니다. `bus_lane_ratio`/`school_zone_count`/`accident_hotspot_count`/`speed_bump_count`/`traffic_volume_aadt`는 이미 dimension에, `traffic_volume_hourly`는 이미 fact에 들어있어(둘 다 `src/pipeline/04`·`05`가 미리 계산) 구 1~3절의 공간조인이 필요 없습니다.

**신규(7차)**: 지금까지 dimension에 있는데도 안 쓰던 **도로망 위상·파생 28개**(`TOPO_COLS`)를 여기서 같이 가져옵니다 — `notebooks/04_tier_split_evaluation.ipynb`가 검증한 목록과 동일합니다.

In [2]:
NUMERIC_COLS = [
    "LANES", "MAX_SPD", "LENGTH", "시간코드",
    "elev_start_m", "elev_mid_m", "elev_end_m", "elev_mean_m", "elev_min_m", "elev_max_m", "elev_delta_m",
    "slope_pct", "abs_slope_pct", "precip_mm",
]
CATEGORICAL_COLS = [
    "ROAD_RANK", "ROAD_TYPE", "ROAD_USE", "MULTI_LINK", "CONNECT",
    "REST_VEH", "REST_W", "REST_H", "C-ITS",
]
# pipeline/04가 이미 0으로 채워둔 외부 컨텍스트(결측 없음)
EXTERNAL_CONTEXT_COLS = ["bus_lane_ratio", "school_zone_count", "accident_hotspot_count", "speed_bump_count"]
# 결측이 정상인 컬럼(전체 매칭 안 됨) — NUMERIC_COLS의 dropna 대상에서 제외하고 LightGBM/XGBoost 기본 결측치 처리에 맡김
SPARSE_NUMERIC_COLS = ["traffic_volume_aadt", "traffic_volume_hourly", "traffic_volume_ratio"]
# 도로망 위상·파생 28개(신규, 7차) — pipeline/04 산출, notebooks/04_tier_split_evaluation.ipynb 검증 목록과 동일
TOPO_COLS = [
    "f_node_node_in_degree", "f_node_incoming_max_spd_mean", "f_node_incoming_lanes_mean",
    "f_node_node_out_degree", "f_node_outgoing_max_spd_mean", "f_node_outgoing_lanes_mean",
    "f_node_node_total_degree",
    "t_node_node_in_degree", "t_node_incoming_max_spd_mean", "t_node_incoming_lanes_mean",
    "t_node_node_out_degree", "t_node_outgoing_max_spd_mean", "t_node_outgoing_lanes_mean",
    "t_node_node_total_degree",
    "upstream_max_spd_mean", "downstream_max_spd_mean", "upstream_lanes_mean", "downstream_lanes_mean",
    "node_complexity", "downstream_speed_delta", "slope_pct_clipped", "slope_outlier_flag",
    "is_uphill", "is_downhill", "steep_uphill", "slope_length_m", "max_spd_x_lanes",
    "length_x_node_complexity",
]
# View-T Level5.5 실측(9차) — 이제 pipeline/04가 정식 제공(28.2%만 커버, 나머지는 NaN 유지)
VIEWT_SUMMARY_COLS = ["annual_avg_speed", "evening_peak_speed", "congest_freq", "congest_time", "congest_expect"]
VIEWT_HOURLY_COLS = [f"avg_speed_h{h}" for h in range(24)]
TARGET = "평균속도"

if CACHE_PATH.exists():
    df = pd.read_parquet(CACHE_PATH)
    print("캐시에서 로드:", CACHE_PATH)
else:
    dim_dtype = {c: str for c in CATEGORICAL_COLS}
    dim_dtype["LINK_ID"] = str
    dimension = pd.read_csv(DIMENSION_PATH, dtype=dim_dtype, encoding="utf-8-sig")
    dim_cols = (["LINK_ID"] + CATEGORICAL_COLS
                + ["LANES", "MAX_SPD", "LENGTH", "elev_start_m", "elev_mid_m", "elev_end_m",
                   "elev_mean_m", "elev_min_m", "elev_max_m", "elev_delta_m", "slope_pct", "abs_slope_pct"]
                + EXTERNAL_CONTEXT_COLS + ["traffic_volume_aadt"] + TOPO_COLS
                + VIEWT_SUMMARY_COLS + VIEWT_HOURLY_COLS)

    # 생활인구(grid_pop/dong_pop, 5절)도 이제 pipeline/05가 fact에 직접 채워줌 - 별도 격자매칭 불필요
    fact = pd.read_csv(
        FACT_PATH, dtype={"LINK_ID": str, "기준일": str}, encoding="utf-8-sig",
        usecols=["LINK_ID", "기준일", "시간코드", TARGET, "precip_mm", "traffic_volume_hourly",
                 "traffic_volume_ratio", "sample_count_5min", "grid_pop", "dong_pop"],
    )

    df = fact.merge(dimension[dim_cols], on="LINK_ID", how="left", validate="many_to_one")

    for col in NUMERIC_COLS + [TARGET]:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("float32")
    for col in EXTERNAL_CONTEXT_COLS:
        df[col] = df[col].astype("float32")
    for col in SPARSE_NUMERIC_COLS:
        df[col] = df[col].astype("float32")
    for col in TOPO_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("float32")
    for col in VIEWT_SUMMARY_COLS + VIEWT_HOURLY_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("float32")
    # sample_count_5min(16차): 결측 없이 모든 관측행에 존재 — sample_weight용, 모델 피처로는 안 씀
    df["sample_count_5min"] = pd.to_numeric(df["sample_count_5min"], errors="coerce").astype("float32")
    df["grid_pop"] = pd.to_numeric(df["grid_pop"], errors="coerce").astype("float32")
    df["dong_pop"] = pd.to_numeric(df["dong_pop"], errors="coerce").astype("float32")

    RENAME_MAP = {"C-ITS": "c_its"}
    df = df.rename(columns=RENAME_MAP)
    CATEGORICAL_COLS = [RENAME_MAP.get(c, c) for c in CATEGORICAL_COLS]

    df = df.dropna(subset=[TARGET] + NUMERIC_COLS)
    df.to_parquet(CACHE_PATH)
    print("새로 만들어서 캐시 저장:", CACHE_PATH)

CATEGORICAL_COLS = [c if c != "C-ITS" else "c_its" for c in CATEGORICAL_COLS]
for col in CATEGORICAL_COLS:
    df[col] = df[col].astype("category")

print("행 수:", len(df), "| 컬럼 수:", df.shape[1])
print("메모리 사용량(MB):", round(df.memory_usage(deep=True).sum() / 1e6, 1))
print()
print("traffic_volume_hourly 결측 비율:", round(df["traffic_volume_hourly"].isna().mean() * 100, 1), "%")
print("traffic_volume_aadt 결측 비율:", round(df["traffic_volume_aadt"].isna().mean() * 100, 1), "%")
print("traffic_volume_ratio 결측 비율:", round(df["traffic_volume_ratio"].isna().mean() * 100, 1), "%")
print("sample_count_5min 결측 비율:", round(df["sample_count_5min"].isna().mean() * 100, 1), "%,",
      "분포(mean/median):", round(df["sample_count_5min"].mean(), 1), "/", df["sample_count_5min"].median())
print("grid_pop/dong_pop 결측 비율:", round(df["grid_pop"].isna().mean() * 100, 1), "% /",
      round(df["dong_pop"].isna().mean() * 100, 1), "%")
print("View-T 실측 결측 비율(annual_avg_speed 기준):", round(df["annual_avg_speed"].isna().mean() * 100, 1), "%")
print("도로망 위상·파생 결측 비율(최대):", round(df[TOPO_COLS].isna().mean().max() * 100, 1), "%")
df.head()


새로 만들어서 캐시 저장: data\interim\notebook_cache\nb12_feature_cache_v9.parquet


행 수: 15094335 | 컬럼 수: 93
메모리 사용량(MB): 5600.0

traffic_volume_hourly 결측 비율: 60.7 %
traffic_volume_aadt 결측 비율: 60.7 %
traffic_volume_ratio 결측 비율: 60.7 %
sample_count_5min 결측 비율: 0.0 %, 분포(mean/median): 8.2 / 8.0


grid_pop/dong_pop 결측 비율: 0.5 % / 0.4 %
View-T 실측 결측 비율(annual_avg_speed 기준): 60.7 %


도로망 위상·파생 결측 비율(최대): 0.0 %


,LINK_ID,기준일,시간코드,sample_count_5min,평균속도,precip_mm,traffic_volume_hourly,traffic_volume_ratio,grid_pop,dong_pop,...,avg_speed_h14,avg_speed_h15,avg_speed_h16,avg_speed_h17,avg_speed_h18,avg_speed_h19,avg_speed_h20,avg_speed_h21,avg_speed_h22,avg_speed_h23
0,1000000301,20260707,1.0,12.0,13.166667,0.0,398.0,0.019915,299.600006,19518.570312,...,22.0,23.0,23.0,22.0,20.0,24.0,25.0,18.0,18.0,17.0
1,1000000301,20260707,2.0,8.0,16.250000,0.0,317.0,0.015862,280.329987,18680.589844,...,22.0,23.0,23.0,22.0,20.0,24.0,25.0,18.0,18.0,17.0
2,1000000301,20260707,3.0,8.0,14.000000,0.0,213.0,0.010658,253.729996,18019.480469,...,22.0,23.0,23.0,22.0,20.0,24.0,25.0,18.0,18.0,17.0
3,1000000301,20260707,4.0,8.0,15.000000,0.0,180.0,0.009007,240.850006,17702.509766,...,22.0,23.0,23.0,22.0,20.0,24.0,25.0,18.0,18.0,17.0
4,1000000301,20260707,5.0,8.0,15.250000,0.0,255.0,0.012760,254.800003,17618.439453,...,22.0,23.0,23.0,22.0,20.0,24.0,25.0,18.0,18.0,17.0


## 2. 파생 피처 생성 (요일/주말/시간 순환 인코딩 + 강수·첨두시)

캐시 이후 단계라 로직을 바꿔도 캐시 무효화가 필요 없습니다.

**신규(7차)**: `src/11_evaluate_speed_model.py`가 쓰는 강수·첨두시 파생 피처를 추가합니다 — `log_precip`, `rain_class`, `is_morning_peak`/`is_evening_peak`, `rain_morning_peak`/`rain_evening_peak`, `rain_x_max_spd`. 첨두시 판정은 **실제 시각**(`시간코드 - 1`) 기준으로 합니다 — `src/11`은 `시간코드`를 보정 없이 그대로 써서 실제로는 한 시간 이른 구간(예: 06~08시)을 "오전 첨두"로 잡고 있었습니다.

In [3]:
df["_date"] = pd.to_datetime(df["기준일"], format="%Y%m%d")
df["dow"] = df["_date"].dt.weekday.astype("category")
df["is_weekend"] = (df["_date"].dt.weekday >= 5).astype("int8")

hour_code = df["시간코드"].astype("float32")
df["hour_sin"] = np.sin(2 * np.pi * hour_code / 24).astype("float32")
df["hour_cos"] = np.cos(2 * np.pi * hour_code / 24).astype("float32")

# 실제 시각 = 시간코드 - 1 (00시대=시간코드01 규칙, pipeline/02 근거) — 첨두시 판정은 이 기준으로
actual_hour = (df["시간코드"].astype("int16") - 1)
df["is_morning_peak"] = actual_hour.isin([7, 8, 9]).astype("int8")
df["is_evening_peak"] = actual_hour.isin([17, 18, 19]).astype("int8")

precip = df["precip_mm"].clip(lower=0)
df["is_rain"] = (precip > 0).astype("int8")
df["log_precip"] = np.log1p(precip).astype("float32")
df["rain_class"] = pd.cut(
    precip, bins=[-np.inf, 0, 1, 5, 10, np.inf],
    labels=["none", "light", "moderate", "heavy", "very_heavy"],
).astype(str).astype("category")
df["rain_morning_peak"] = (df["is_rain"] * df["is_morning_peak"]).astype("int8")
df["rain_evening_peak"] = (df["is_rain"] * df["is_evening_peak"]).astype("int8")
df["rain_x_max_spd"] = (df["log_precip"] * df["MAX_SPD"]).astype("float32")

df = df.drop(columns=["_date"])


## 3. 인접 실측 이웃 속도

**목적**: 지금 만드는 피처들은 전부 "미관측 링크에도 쓸 수 있는 범용 피처" 후보를 찾는 작업입니다(관측 링크 자체는 나중에 Model A처럼 자기 이력을 쓰면 되니 여기서는 다루지 않습니다). 이번엔 "이 링크와 도로망으로 맞닿은 링크가 **그 시각에 실제로 관측된 값**"을 피처로 추가합니다(이웃의 예측값이 아니라 실측값 — 순환 의존 문제 없음).

사전 검증(별도 확인): 관측 링크의 97.0%가 관측된 1홉 이웃을 갖고 있고, 자기 실제속도와 이웃의 실제속도 상관계수는 0.685였습니다. 미관측 링크 기준으로는 75.3%가 1홉 안에 실측 이웃을 갖습니다.

**계산 방식**: `F_NODE`/`T_NODE` 위상으로 1홉 인접 그래프를 만들고, 링크 x 시간대(LINK_ID x tm) 속도 행렬에 희소 인접행렬을 곱해 "1홉 관측 이웃들의 그 시각 평균 실제속도"를 한 번에 계산합니다(결측 이웃은 자동으로 평균에서 제외). 1홉 안에 관측 이웃이 없는 링크는 NaN으로 남겨 LightGBM 기본 결측 처리에 맡깁니다.

**주의**: 여기서 만드는 `nearest_obs_neighbor_speed`는 전체 데이터 기준 계산이라 그대로 평가에 쓰면 안 됩니다(같은 링크가 train/test 양쪽에 실측값을 흘려보내 누수됩니다) — 9절 GroupKFold에서 폴드마다 train 링크만으로 다시 계산합니다(`build_neighbor_feature_for_allowed`). 여기서 만든 버전은 이후 파생 피처 계산과 데이터 탐색용입니다.

In [4]:
import geopandas as gpd
from collections import defaultdict
from scipy.sparse import csr_matrix

seoul_link_topo = gpd.read_file(SEOUL_LINK_SHP)[["LINK_ID", "F_NODE", "T_NODE"]]
seoul_link_topo["LINK_ID"] = seoul_link_topo["LINK_ID"].astype(str)

node_links = defaultdict(set)
for r in seoul_link_topo.itertuples(index=False):
    node_links[r.F_NODE].add(r.LINK_ID)
    node_links[r.T_NODE].add(r.LINK_ID)
link_neighbors = defaultdict(set)
for node, links in node_links.items():
    for l in links:
        link_neighbors[l].update(links - {l})

df["tm_key"] = df["기준일"].astype(str) + "_" + df["시간코드"].astype(int).astype(str)

link_ids_sorted = np.sort(df["LINK_ID"].unique())
link_idx = {l: i for i, l in enumerate(link_ids_sorted)}
observed_set = set(link_ids_sorted)
n_links = len(link_ids_sorted)

rows_idx, cols_idx = [], []
for l in link_ids_sorted:
    i = link_idx[l]
    for n in link_neighbors.get(l, ()):
        if n in observed_set:
            rows_idx.append(i)
            cols_idx.append(link_idx[n])
A = csr_matrix((np.ones(len(rows_idx), dtype="float32"), (rows_idx, cols_idx)), shape=(n_links, n_links))
print("인접(관측-관측) 엣지 수:", len(rows_idx), "| 1홉 관측 이웃이 0개인 링크:",
      (np.asarray(A.sum(axis=1)).ravel() == 0).sum())

speed_wide = df.pivot_table(index="LINK_ID", columns="tm_key", values=TARGET, aggfunc="mean")
speed_wide = speed_wide.reindex(link_ids_sorted)

valid = (~speed_wide.isna()).to_numpy(dtype="float32")
filled = np.nan_to_num(speed_wide.to_numpy(dtype="float32"), nan=0.0)

numerator = A @ filled
denom = A @ valid
with np.errstate(invalid="ignore", divide="ignore"):
    neighbor_mean = numerator / denom
neighbor_mean[denom == 0] = np.nan

tm_columns = list(speed_wide.columns)  # 4절(2홉 확장)·9절(폴드별 재계산)에서 재사용

neighbor_long = pd.DataFrame(neighbor_mean, index=speed_wide.index, columns=speed_wide.columns) \
    .reset_index().melt(id_vars="LINK_ID", var_name="tm_key", value_name="nearest_obs_neighbor_speed")
df = df.merge(neighbor_long, on=["LINK_ID", "tm_key"], how="left")
df = df.drop(columns=["tm_key"])

# filled/valid(링크x시간 실측 행렬)와 link_neighbors/link_idx/A는 4절(2홉 확장)과 9절(GroupKFold,
# 폴드별 이웃 재계산)에서 다시 쓰므로 지우지 않음 — speed_wide와 중간 계산값만 정리
del speed_wide, numerator, denom, neighbor_mean, neighbor_long
gc.collect()

print("nearest_obs_neighbor_speed 결측 비율:", round(df["nearest_obs_neighbor_speed"].isna().mean() * 100, 1), "%")
print(df["nearest_obs_neighbor_speed"].describe())


인접(관측-관측) 엣지 수: 145764 | 1홉 관측 이웃이 0개인 링크: 1334


nearest_obs_neighbor_speed 결측 비율: 3.0 %


count    1.464761e+07
mean     2.300422e+01
std      1.039420e+01
min      1.000000e+00
25%      1.662500e+01
50%      2.064583e+01
75%      2.613750e+01
max      1.079375e+02
Name: nearest_obs_neighbor_speed, dtype: float64


## 4. (신규) 2홉 이웃 확장 — 블렌딩(10차) vs 폴백(11차)

3절의 `nearest_obs_neighbor_speed`는 1홉 이내에 관측 이웃이 없으면(C 계층) 결측으로 남습니다. 사전 확인(그래프 구조만으로 GroupKFold 3-fold 커버리지 체크, 모델 학습 없이): C 계층(1홉 이웃 없음, test 링크의 5.5%) 중 **91.1%가 2홉까지 넓히면 관측 이웃을 찾습니다** — 2홉까지 넓혀도 못 찾는 "진짜 C"는 전체 test의 0.5%까지 줄어듭니다(서울 도로망이 촘촘해서 교차로마다 평균 3~4개 링크가 붙어있기 때문).

**10차(블렌딩)**: 1홉 이웃(가중치 1.0)과 2홉 전용 이웃(가중치 0.5)을 합친 가중평균으로 `nearest_obs_neighbor_speed_2hop`을 계산해 1홉 버전을 통째로 교체했습니다. **실행 결과**: C 계층 커버리지는 크게 개선됐지만(283,255행→24,560행), 원래 1홉이 있던 475만 행에도 2홉이 섞여 들어가면서 B 계층 R²가 0.741→0.731로 떨어졌습니다.

**11차(폴백)**: 블렌딩 대신 **1홉이 있으면 1홉만, 없을 때만 2홉 순수평균으로 대체**하는 `nearest_obs_neighbor_speed_fallback`을 만들었습니다. **실행 결과(정정)**: 예상과 달리 B 계층 R²는 10차와 동일(0.731)하게 나왔습니다 — 원인을 분석해보니 "블렌딩이 B를 오염시킨다"는 최초 가설이 아니라, **B의 정의 자체가(1홉 또는 2홉 있으면 B로) 넓어지면서 새로 편입된 25만 행의 평균이 원래 낮기 때문**이었습니다(방식과 무관하게 발생하는 산술적 결과). 대신 MAE는 11차가 소폭 더 낫습니다(4.428 vs 4.468) — 폴백이 실질적으로 조금 더 나은 설계입니다.

**주의**: 여기서 만드는 것도 3절과 마찬가지로 전체 데이터 기준 계산이라 그대로 평가에 쓰면 누수입니다 — 9절 GroupKFold에서 폴드마다 train 링크만으로 다시 계산합니다.

In [5]:
link_neighbors_2hop = {}
for l in link_ids_sorted:
    one_hop = link_neighbors.get(l, set())
    two_hop = set()
    for n in one_hop:
        two_hop |= link_neighbors.get(n, set())
    two_hop -= one_hop
    two_hop.discard(l)
    link_neighbors_2hop[l] = two_hop

# --- 10차: 1홉(가중치 1.0) + 2홉전용(가중치 0.5) 블렌딩 ---
W1, W2 = 1.0, 0.5

rows2, cols2 = [], []
for l in link_ids_sorted:
    i = link_idx[l]
    for n in link_neighbors_2hop.get(l, ()):
        if n in observed_set:
            rows2.append(i)
            cols2.append(link_idx[n])
A2_only = csr_matrix((np.full(len(rows2), W2, dtype="float32"), (rows2, cols2)), shape=(n_links, n_links))

A_2hop = A * W1 + A2_only  # A는 3절의 1홉 관측-관측 인접행렬(가중치 1.0)
num2 = A_2hop @ filled
den2 = A_2hop @ valid
with np.errstate(invalid="ignore", divide="ignore"):
    neighbor_mean_2hop = num2 / den2
neighbor_mean_2hop[den2 == 0] = np.nan

neighbor_long_2hop = pd.DataFrame(neighbor_mean_2hop, index=link_ids_sorted, columns=tm_columns) \
    .reset_index().rename(columns={"index": "LINK_ID"}) \
    .melt(id_vars="LINK_ID", var_name="tm_key", value_name="nearest_obs_neighbor_speed_2hop")

df["tm_key"] = df["기준일"].astype(str) + "_" + df["시간코드"].astype(int).astype(str)
df = df.merge(neighbor_long_2hop, on=["LINK_ID", "tm_key"], how="left")

has_1hop_link = np.asarray(A.sum(axis=1)).ravel() > 0
has_2hop_extra_link = np.asarray(A2_only.sum(axis=1)).ravel() > 0
newly_covered_link = (~has_1hop_link) & has_2hop_extra_link
still_uncovered_link = (~has_1hop_link) & ~has_2hop_extra_link

print(f"1홉 이웃 없는 링크: {(~has_1hop_link).sum():,}")
print(f"  -> 2홉으로 새로 커버되는 링크: {newly_covered_link.sum():,} "
      f"({newly_covered_link.sum() / max((~has_1hop_link).sum(), 1):.1%})")
print(f"  -> 그래도 안 커버되는 링크: {still_uncovered_link.sum():,}")
print()
print("nearest_obs_neighbor_speed_2hop(10차, 블렌딩) 결측 비율:",
      round(df["nearest_obs_neighbor_speed_2hop"].isna().mean() * 100, 1), "%")

# --- 11차: 2홉전용 순수평균(가중치 1.0, 블렌딩 아님) -> 1홉 없을 때만 폴백으로 사용 ---
rows2b, cols2b = [], []
for l in link_ids_sorted:
    i = link_idx[l]
    for n in link_neighbors_2hop.get(l, ()):
        if n in observed_set:
            rows2b.append(i)
            cols2b.append(link_idx[n])
A2_pure = csr_matrix((np.ones(len(rows2b), dtype="float32"), (rows2b, cols2b)), shape=(n_links, n_links))
num2b = A2_pure @ filled
den2b = A2_pure @ valid
with np.errstate(invalid="ignore", divide="ignore"):
    neighbor_mean_2hop_only = num2b / den2b
neighbor_mean_2hop_only[den2b == 0] = np.nan

neighbor_long_2hop_only = pd.DataFrame(neighbor_mean_2hop_only, index=link_ids_sorted, columns=tm_columns) \
    .reset_index().rename(columns={"index": "LINK_ID"}) \
    .melt(id_vars="LINK_ID", var_name="tm_key", value_name="nearest_obs_neighbor_speed_2hop_only")

df = df.merge(neighbor_long_2hop_only, on=["LINK_ID", "tm_key"], how="left")
df = df.drop(columns=["tm_key"])

df["nearest_obs_neighbor_speed_fallback"] = df["nearest_obs_neighbor_speed"].fillna(df["nearest_obs_neighbor_speed_2hop_only"])

used_fallback = df["nearest_obs_neighbor_speed"].isna() & df["nearest_obs_neighbor_speed_2hop_only"].notna()
print()
print(f"1홉 없어서 2홉 폴백을 실제로 쓴 행 비율(11차): {used_fallback.mean():.2%}")
print("nearest_obs_neighbor_speed_fallback(11차) 결측 비율:",
      round(df["nearest_obs_neighbor_speed_fallback"].isna().mean() * 100, 1), "%")

1홉 이웃 없는 링크: 1,334
  -> 2홉으로 새로 커버되는 링크: 1,206 (90.4%)
  -> 그래도 안 커버되는 링크: 128

nearest_obs_neighbor_speed_2hop(10차, 블렌딩) 결측 비율: 0.3 %



1홉 없어서 2홉 폴백을 실제로 쓴 행 비율(11차): 2.68%
nearest_obs_neighbor_speed_fallback(11차) 결측 비율: 0.3 %


## 5. 생활인구 — 격자/행정동 (pipeline/05 네이티브)

`grid_pop`/`dong_pop`은 이제 `pipeline/05_build_speed_fact.py`가 `seoul_link_speed_fact.csv`에 직접 채워 넣는 정식 컬럼입니다(이번에 파이프라인으로 이관, 이전엔 이 노트북이 매번 `match.shp`+격자/행정동 CSV를 직접 읽어 만들던 실험용 데이터였습니다). 매칭 로직(링크→250m격자→대표행정동, `시간코드-1`=실제시 보정)은 `pipeline/05`의 `build_population_features()`를 참고하세요. 여기서는 이미 1절에서 로드된 값을 그대로 확인만 합니다.

In [6]:
# hour(=시간코드-1, 실제시 0~23) - 6절 View-T 시간대별 매칭에서 재사용
df["hour"] = (df["시간코드"].astype(int) - 1)

print(f"grid_pop 결측 없는 행 비율: {df['grid_pop'].notna().mean():.4f}")
print(f"dong_pop 결측 없는 행 비율: {df['dong_pop'].notna().mean():.4f}")
df[["grid_pop", "dong_pop"]].describe()

grid_pop 결측 없는 행 비율: 0.9951
dong_pop 결측 없는 행 비율: 0.9958


,grid_pop,dong_pop
count,1.502026e+07,1.503117e+07
mean,1.775540e+03,3.164569e+04
std,1.826875e+03,2.011730e+04
min,0.000000e+00,2.489320e+03
25%,3.869800e+02,2.023400e+04
50%,1.429770e+03,2.753769e+04
75%,2.583510e+03,3.774350e+04
max,5.454269e+04,2.274385e+05


## 6. View-T 실측 평균속도·혼잡지표 (pipeline/04 네이티브)

`annual_avg_speed`/`evening_peak_speed`/`avg_speed_h0~23`/`congest_freq`/`congest_time`/`congest_expect`도 이제 `pipeline/04_build_link_dimension.py`가 `seoul_link_dimension.csv`에 직접 채워 넣습니다(이전엔 이 노트북이 매번 `AverageSpeed(LINK).xlsx`/`CongestIndex(LINK).xlsx`를 직접 읽어 만들던 실험용 데이터였습니다). Level5.5(주요도로망) 17,929개 링크만 커버(C 계층 1.1%)하는 건 동일합니다 - `annual_avg_speed`는 실측 링크평균속도와 상관계수 0.787(최강 단일 피처 후보)이고 `MAX_SPD`와의 상관은 0.460뿐이라 완전 중복은 아닙니다.

시간대별 값(`avg_speed_hourly`)만 이 노트북에서 만듭니다 - dimension에는 `avg_speed_h0~h23` 24개 컬럼으로 넓게 저장돼 있어서, 그 행의 실제 시간(`hour`)에 해당하는 값만 골라 씁니다.

In [7]:
hour_cols = [f"avg_speed_h{h}" for h in range(24)]
hour_mat = df[hour_cols].to_numpy(dtype="float32")
actual_hour = df["hour"].clip(0, 23).to_numpy()
df["avg_speed_hourly"] = hour_mat[np.arange(len(df)), actual_hour]
df = df.drop(columns=hour_cols)

VIEWT_COLS = ["annual_avg_speed", "evening_peak_speed", "avg_speed_hourly", "congest_freq", "congest_time", "congest_expect"]
for col in VIEWT_COLS:
    df[col] = df[col].astype("float32")

print(f"행 기준 매칭률: {df['annual_avg_speed'].notna().mean():.1%}")
df[VIEWT_COLS].describe()

행 기준 매칭률: 39.3%


,annual_avg_speed,evening_peak_speed,avg_speed_hourly,congest_freq,congest_time,congest_expect
count,5.938480e+06,5.934112e+06,5.935218e+06,5.766448e+06,5.766448e+06,5766448.0
mean,2.989737e+01,2.753609e+01,3.166991e+01,5.169582e+01,6.928394e+01,0.0
std,9.902841e+00,9.825967e+00,1.167102e+01,2.492275e+01,2.052282e+01,0.0
min,5.000000e+00,5.000000e+00,3.000000e+00,0.000000e+00,0.000000e+00,0.0
25%,2.300000e+01,2.100000e+01,2.400000e+01,3.400000e+01,5.700000e+01,0.0
50%,2.800000e+01,2.566670e+01,3.000000e+01,4.800000e+01,7.050000e+01,0.0
75%,3.400000e+01,3.200000e+01,3.700000e+01,6.900000e+01,8.500000e+01,0.0
max,9.600000e+01,9.666670e+01,1.500000e+02,1.000000e+02,1.000000e+02,0.0


## 7. (신규, 12차) 코리도 규모 + 도심거리

`experiments/build_structural_features.py`의 아이디어를 재사용합니다 — 이력이 없는 링크(C 계층)는 자기 과거를 못 쓰니, "이 도로가 구조적으로 어떤 도로인가"를 도로망 자체에서 뽑아 대신 씁니다. 라벨(속도)을 전혀 안 써서 누수가 없고 63,548개 링크 100% 커버됩니다.

- **코리도 규모**(`rn_links`/`rn_len`/`rn_lanes`/`rn_maxspd`): `ROAD_NAME`(도로명)으로 묶어서 "이 도로 전체가 몇 개 링크·몇 km·평균 몇 차로·평균 제한속도"인지 집계합니다. 도로명이 없는 링크는 "단독 링크"로 취급(자기 자신 값 그대로).
- **도심거리**(`dist_cbd`): 서울시청 부근 좌표(198000, 552000, EPSG:5179 ITRF2000 중부원점) 기준, 링크 중심점까지의 유클리드 거리입니다.

원본 스크립트는 `MOCT_NODE.shp`/`TURNINFO.dbf`로 회전제한·교차로유형까지 뽑지만, 이번엔 사용자가 요청한 코리도+도심거리까지만 넣습니다.

In [8]:
CBD_X, CBD_Y = 198000.0, 552000.0  # 서울시청 부근 좌표(EPSG:5179)

road_name_raw = pd.read_csv(
    DIMENSION_PATH, dtype={"LINK_ID": str, "ROAD_NAME": str},
    usecols=["LINK_ID", "ROAD_NAME", "LANES", "MAX_SPD", "LENGTH"], encoding="utf-8-sig",
)
for col in ["LANES", "MAX_SPD", "LENGTH"]:
    road_name_raw[col] = pd.to_numeric(road_name_raw[col], errors="coerce")

valid_name = road_name_raw["ROAD_NAME"].notna() & ~road_name_raw["ROAD_NAME"].isin(["", "-", "None"])
rn = road_name_raw[valid_name].groupby("ROAD_NAME").agg(
    rn_links=("LINK_ID", "size"), rn_len=("LENGTH", "sum"),
    rn_lanes=("LANES", "mean"), rn_maxspd=("MAX_SPD", "mean"),
).reset_index()
print(f"이름 있는 도로: {valid_name.sum():,} / {len(road_name_raw):,} 링크 | 고유 도로명 {rn.shape[0]:,}")

corridor = road_name_raw.merge(rn, on="ROAD_NAME", how="left")
corridor["rn_links"] = corridor["rn_links"].fillna(1)
corridor["rn_len"] = corridor["rn_len"].fillna(corridor["LENGTH"])
corridor["rn_lanes"] = corridor["rn_lanes"].fillna(corridor["LANES"])
corridor["rn_maxspd"] = corridor["rn_maxspd"].fillna(corridor["MAX_SPD"])
corridor = corridor[["LINK_ID", "rn_links", "rn_len", "rn_lanes", "rn_maxspd"]]

link_centroid = gpd.read_file(SEOUL_LINK_SHP)[["LINK_ID", "geometry"]]
link_centroid["LINK_ID"] = link_centroid["LINK_ID"].astype(str)
link_centroid["geometry"] = link_centroid.geometry.centroid
link_centroid = link_centroid.to_crs("EPSG:5179")
corridor_cbd = pd.DataFrame({
    "LINK_ID": link_centroid["LINK_ID"],
    "dist_cbd": np.sqrt((link_centroid.geometry.x - CBD_X) ** 2 + (link_centroid.geometry.y - CBD_Y) ** 2),
})

df = df.merge(corridor, on="LINK_ID", how="left")
df = df.merge(corridor_cbd, on="LINK_ID", how="left")

CORRIDOR_COLS = ["rn_links", "rn_len", "rn_lanes", "rn_maxspd", "dist_cbd"]
for col in CORRIDOR_COLS:
    df[col] = df[col].astype("float32")

print(f"코리도/도심거리 결측 비율(최대): {df[CORRIDOR_COLS].isna().mean().max() * 100:.1f} %")
df[CORRIDOR_COLS].describe()

이름 있는 도로: 62,681 / 63,548 링크 | 고유 도로명 2,986


코리도/도심거리 결측 비율(최대): 0.0 %


,rn_links,rn_len,rn_lanes,rn_maxspd,dist_cbd
count,1.509434e+07,1.509434e+07,1.509434e+07,1.509434e+07,1.509434e+07
mean,9.911013e+01,1.200398e+04,2.014063e+00,4.174430e+01,1.589861e+06
std,1.055803e+02,2.144441e+04,8.398191e-01,9.916969e+00,3.064003e+05
min,1.000000e+00,5.568372e+00,1.000000e+00,1.000000e+01,1.574984e+06
25%,2.500000e+01,2.068637e+03,1.112903e+00,3.016949e+01,1.584434e+06
50%,7.100000e+01,6.354091e+03,2.000000e+00,4.445652e+01,1.589287e+06
75%,1.240000e+02,1.137582e+04,2.685567e+00,4.966102e+01,1.594724e+06
max,6.010000e+02,1.617842e+05,8.000000e+00,8.700000e+01,1.606804e+06


## 7-1. (신규, 14차) 회전제한/교차로유형

`experiments/build_structural_features.py`의 나머지 피처(코리도/도심거리는 12차에서 이미 반영). `MOCT_NODE.shp`(전국)의 `NODE_TYPE`(101=교차로), `TURNINFO.dbf`(노드별 회전제한 건수)를 링크의 F_NODE/T_NODE에 매칭합니다. 라벨을 전혀 안 써서 누수 없이 63,548개 링크 100% 커버되고, 코리도처럼 C 계층에도 새 신호를 줄 수 있는지 확인합니다.

In [9]:
MOCT_NODE_SHP = RAW_DIR / "road" / "MOCT_NODE.shp"
TURNINFO_DBF = RAW_DIR / "road" / "TURNINFO.dbf"

node_info = gpd.read_file(MOCT_NODE_SHP)[["NODE_ID", "NODE_TYPE"]]
node_info["NODE_ID"] = node_info["NODE_ID"].astype(str)
node_info["NODE_TYPE"] = pd.to_numeric(node_info["NODE_TYPE"], errors="coerce")
node_type_lookup = node_info.set_index("NODE_ID")["NODE_TYPE"]

turn_info = gpd.read_file(TURNINFO_DBF)[["NODE_ID"]]
turn_info["NODE_ID"] = turn_info["NODE_ID"].astype(str)
turn_n_lookup = turn_info["NODE_ID"].value_counts()

link_nodes = gpd.read_file(SEOUL_LINK_SHP)[["LINK_ID", "F_NODE", "T_NODE"]]
link_nodes["LINK_ID"] = link_nodes["LINK_ID"].astype(str)

f_type = node_type_lookup.reindex(link_nodes["F_NODE"]).to_numpy()
t_type = node_type_lookup.reindex(link_nodes["T_NODE"]).to_numpy()

turn_df = pd.DataFrame({
    "LINK_ID": link_nodes["LINK_ID"],
    "turn_n_f": turn_n_lookup.reindex(link_nodes["F_NODE"]).fillna(0).to_numpy(),
    "turn_n_t": turn_n_lookup.reindex(link_nodes["T_NODE"]).fillna(0).to_numpy(),
    "is_cross_f": (f_type == 101).astype("int8"),
    "is_cross_t": (t_type == 101).astype("int8"),
})

df = df.merge(turn_df, on="LINK_ID", how="left")

TURN_COLS = ["turn_n_f", "turn_n_t", "is_cross_f", "is_cross_t"]
for col in TURN_COLS:
    df[col] = df[col].astype("float32")

print(f"회전제한/교차로유형 결측 비율(최대): {df[TURN_COLS].isna().mean().max() * 100:.1f} %")
print(f"교차로 끝 링크 비율: {((df['is_cross_f']==1)|(df['is_cross_t']==1)).mean():.1%}")
df[TURN_COLS].describe()

회전제한/교차로유형 결측 비율(최대): 0.0 %
교차로 끝 링크 비율: 94.0%


,turn_n_f,turn_n_t,is_cross_f,is_cross_t
count,1.509434e+07,1.509434e+07,1.509434e+07,1.509434e+07
mean,1.710300e-01,1.711470e-01,9.004958e-01,9.008855e-01
std,4.150480e-01,4.189026e-01,2.870954e-01,2.866440e-01
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00
50%,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00
75%,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00
max,6.000000e+00,6.000000e+00,1.000000e+00,1.000000e+00


## 8. 피처 세트 정의 (8차~15차)

- **8~14차**: 이전과 동일(위 4~7절 참고). **13차가 지금까지 최선**(R²=0.737, 59개 피처) — 14차(추가 정리+회전제한/교차로)는 C 계층이 오히려 악화돼 채택하지 않았습니다.
- **15차**: 피처는 13차와 완전히 동일(`FEATURE_COLS_13CHA`), **LightGBM 대신 XGBoost**로 학습해서 알고리즘 자체를 바꿨을 때 성능이 달라지는지 확인합니다. `notebooks/01_speed_model_baseline.ipynb`의 무작위분할 베이스라인에서 XGBoost가 LightGBM보다 나았던 사례(R² 0.610→0.632)가 있어, 우리 GroupKFold 기준에서도 재현되는지 검증합니다.

여덟 구성 모두 B/C 분리모델(이웃속도 포함 모델 + 정적 전용 모델)로 평가합니다.

In [10]:
SPARSE_NUMERIC_COLS = SPARSE_NUMERIC_COLS + ["nearest_obs_neighbor_speed"]
POPULATION_COLS = ["grid_pop", "dong_pop"]
RAIN_PEAK_NUMERIC_COLS = [
    "is_rain", "log_precip", "is_morning_peak", "is_evening_peak",
    "rain_morning_peak", "rain_evening_peak", "rain_x_max_spd",
]
RAIN_PEAK_CATEGORICAL_COLS = ["rain_class"]
DERIVED_COLS = (
    ["dow", "is_weekend", "hour_sin", "hour_cos"]
    + EXTERNAL_CONTEXT_COLS
    + SPARSE_NUMERIC_COLS
)

# 8차: 7차까지의 전체 피처(도로기본/지형/시간 + 외부컨텍스트 + 이웃속도 + 위상 + 강수·첨두파생 + 생활인구)
FULL_FEATURE_COLS = (
    NUMERIC_COLS
    + CATEGORICAL_COLS
    + DERIVED_COLS
    + TOPO_COLS
    + POPULATION_COLS
    + RAIN_PEAK_NUMERIC_COLS
    + RAIN_PEAK_CATEGORICAL_COLS
)
FULL_CATEGORICAL_COLS = CATEGORICAL_COLS + RAIN_PEAK_CATEGORICAL_COLS

# 9차: 8차 + View-T 실측 평균속도·혼잡지표
VIEWT_FEATURE_COLS = FULL_FEATURE_COLS + VIEWT_COLS

# 10차: 9차와 동일하되 이웃속도만 1홉 -> 1홉+2홉 블렌딩으로 교체
FEATURE_COLS_10CHA = [
    "nearest_obs_neighbor_speed_2hop" if c == "nearest_obs_neighbor_speed" else c
    for c in VIEWT_FEATURE_COLS
]

# 11차: 9차와 동일하되 이웃속도만 1홉 우선 + 2홉 폴백으로 교체
FEATURE_COLS_11CHA = [
    "nearest_obs_neighbor_speed_fallback" if c == "nearest_obs_neighbor_speed" else c
    for c in VIEWT_FEATURE_COLS
]

# 12차: 11차 + 코리도/도심거리(7절) + 위상·고도 중복 컬럼 제거(notebooks/14 상관분석 근거)
REDUNDANT_TOPO_COLS = [
    "f_node_node_total_degree", "t_node_node_total_degree",
    "upstream_max_spd_mean", "upstream_lanes_mean",
    "downstream_max_spd_mean", "downstream_lanes_mean",
]
REDUNDANT_ELEV_COLS = ["elev_start_m", "elev_mid_m", "elev_end_m", "elev_min_m", "elev_max_m"]
TOPO_COLS_PRUNED = [c for c in TOPO_COLS if c not in REDUNDANT_TOPO_COLS]

FEATURE_COLS_12CHA = (
    [c for c in FEATURE_COLS_11CHA if c not in REDUNDANT_TOPO_COLS and c not in REDUNDANT_ELEV_COLS]
    + CORRIDOR_COLS
)

# 13차: 12차에서 gain 0(0.0000~0.0001)인 13개 컬럼 제거(11b33eca 실행에서 확인된 실측 gain 기준)
DEAD_13 = [
    "is_rain", "log_precip", "is_morning_peak", "is_evening_peak",
    "rain_morning_peak", "rain_evening_peak", "rain_x_max_spd", "rain_class",
    "is_uphill", "is_downhill", "steep_uphill", "slope_outlier_flag",
    "congest_expect",
]
FEATURE_COLS_13CHA = [c for c in FEATURE_COLS_12CHA if c not in DEAD_13]
CATEGORICAL_COLS_13CHA = [c for c in FULL_CATEGORICAL_COLS if c not in DEAD_13]

# 14차: 13차에서 gain 최하위권 9개 추가 제거, log_precip(강수 파생 1개)만 복원, 회전제한/교차로유형 4개 추가
PRUNE_14 = [
    "REST_W", "c_its", "MULTI_LINK", "slope_pct_clipped",
    "t_node_outgoing_max_spd_mean", "f_node_incoming_lanes_mean",
    "congest_freq", "rn_links", "precip_mm",
]
FEATURE_COLS_14CHA = (
    [c for c in FEATURE_COLS_13CHA if c not in PRUNE_14]
    + ["log_precip"]
    + TURN_COLS
)
CATEGORICAL_COLS_14CHA = [c for c in CATEGORICAL_COLS_13CHA if c not in PRUNE_14]

# 16차: 13차 + is_cross_t(14차에서 유일하게 검증된 순수 이득)
# traffic_volume_ratio는 SPARSE_NUMERIC_COLS -> DERIVED_COLS 경유로 13차에 이미 포함돼 있어 따로 안 더함(중복 컬럼 방지)
# sample_count_5min은 피처가 아니라 학습 시 sample_weight로만 씀(9e7693e1 아래 CONFIGS 참고)
FEATURE_COLS_16CHA = FEATURE_COLS_13CHA + ["is_cross_t"]
CATEGORICAL_COLS_16CHA = CATEGORICAL_COLS_13CHA

print("8차(FULL_FEATURE_COLS) 피처 수:", len(FULL_FEATURE_COLS))
print("9차(VIEWT_FEATURE_COLS) 피처 수:", len(VIEWT_FEATURE_COLS))
print("10차(FEATURE_COLS_10CHA) 피처 수:", len(FEATURE_COLS_10CHA))
print("11차(FEATURE_COLS_11CHA) 피처 수:", len(FEATURE_COLS_11CHA))
print("12차(FEATURE_COLS_12CHA) 피처 수:", len(FEATURE_COLS_12CHA))
print("13차(FEATURE_COLS_13CHA) 피처 수:", len(FEATURE_COLS_13CHA))
print("14차(FEATURE_COLS_14CHA) 피처 수:", len(FEATURE_COLS_14CHA),
      f"(13차에서 {len(PRUNE_14)}개 제거 + log_precip 1개 복원 + 회전제한/교차로 {len(TURN_COLS)}개 추가)")
print("16차(FEATURE_COLS_16CHA) 피처 수:", len(FEATURE_COLS_16CHA),
      "(13차 + is_cross_t + traffic_volume_ratio, sample_weight는 sample_count_5min 별도 적용)")
df[DERIVED_COLS + POPULATION_COLS + RAIN_PEAK_NUMERIC_COLS + RAIN_PEAK_CATEGORICAL_COLS].describe(include="all")

8차(FULL_FEATURE_COLS) 피처 수: 73
9차(VIEWT_FEATURE_COLS) 피처 수: 79
10차(FEATURE_COLS_10CHA) 피처 수: 79
11차(FEATURE_COLS_11CHA) 피처 수: 79
12차(FEATURE_COLS_12CHA) 피처 수: 73
13차(FEATURE_COLS_13CHA) 피처 수: 60
14차(FEATURE_COLS_14CHA) 피처 수: 56 (13차에서 9개 제거 + log_precip 1개 복원 + 회전제한/교차로 4개 추가)
16차(FEATURE_COLS_16CHA) 피처 수: 61 (13차 + is_cross_t + traffic_volume_ratio, sample_weight는 sample_count_5min 별도 적용)


,dow,is_weekend,hour_sin,hour_cos,bus_lane_ratio,school_zone_count,accident_hotspot_count,speed_bump_count,traffic_volume_aadt,traffic_volume_hourly,...,grid_pop,dong_pop,is_rain,log_precip,is_morning_peak,is_evening_peak,rain_morning_peak,rain_evening_peak,rain_x_max_spd,rain_class
count,15094335.0,1.509434e+07,1.509434e+07,1.509434e+07,1.509434e+07,1.509434e+07,1.509434e+07,1.509434e+07,5.938480e+06,5.938480e+06,...,1.502026e+07,1.503117e+07,1.509434e+07,1.509434e+07,1.509434e+07,1.509434e+07,1.509434e+07,1.509434e+07,1.509434e+07,15094335
unique,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5
top,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,none
freq,2157217.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11536235
mean,NaN,2.857848e-01,-2.138451e-05,7.420300e-06,6.608189e-02,6.103236e-01,3.469555e-01,2.866067e-01,2.066972e+04,8.612402e+02,...,1.775540e+03,3.164569e+04,2.357242e-01,6.981273e-01,1.249963e-01,1.250046e-01,3.565914e-02,1.768922e-02,2.934275e+01,NaN
std,NaN,4.517874e-01,7.071049e-01,7.071076e-01,2.248980e-01,8.294121e-01,1.594318e+00,8.003197e-01,1.513154e+04,7.628351e+02,...,1.826875e+03,2.011730e+04,4.244506e-01,1.213060e+00,3.307147e-01,3.307242e-01,1.854389e-01,1.318192e-01,5.454292e+01,NaN
min,NaN,0.000000e+00,-1.000000e+00,-1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.093000e+03,0.000000e+00,...,0.000000e+00,2.489320e+03,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,NaN
25%,NaN,0.000000e+00,-7.071067e-01,-7.071068e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.115400e+04,3.570000e+02,...,3.869800e+02,2.023400e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,NaN
50%,NaN,0.000000e+00,-8.742278e-08,1.192488e-08,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.748900e+04,6.600000e+02,...,1.429770e+03,2.753769e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,NaN
75%,NaN,1.000000e+00,7.071068e-01,7.071070e-01,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,2.567400e+04,1.144000e+03,...,2.583510e+03,3.774350e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,NaN


### 8-1. (신규) 그룹별 상관계수 — 추가로 줄일 피처 확인

모델 재학습 없이(GroupKFold 전) 순수 상관분석만 합니다. 그룹 내에서 `|r|>=0.8`인 쌍은 사실상 같은 정보를 중복 저장하는 것이라 다음 차수 제거 후보입니다. 코리도/도심거리가 기존 피처(자기 자신의 `MAX_SPD`/`LANES` 등)와 얼마나 겹치는지도 같이 확인합니다.

In [11]:
HIGH_CORR_THRESHOLD = 0.8

CORR_GROUPS = {
    "도로기본(수치, 9개)": [c for c in NUMERIC_COLS if c not in REDUNDANT_ELEV_COLS],
    "위상·파생(7차, 22개)": TOPO_COLS_PRUNED,
    "생활인구(6차, 2개)": POPULATION_COLS,
    "강수·첨두파생(7차, 수치 7개)": RAIN_PEAK_NUMERIC_COLS,
    "ViewT(9차, 6개)": VIEWT_COLS,
    "코리도/도심거리(12차, 5개)": CORRIDOR_COLS,
}

for group_name, cols in CORR_GROUPS.items():
    corr = df[cols].corr()
    pairs = []
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            r = corr.iloc[i, j]
            if pd.notna(r) and abs(r) >= HIGH_CORR_THRESHOLD:
                pairs.append((cols[i], cols[j], round(float(r), 3)))
    print(f"=== {group_name} — |r|>=0.8 쌍 ===")
    if pairs:
        for a, b, r in sorted(pairs, key=lambda x: -abs(x[2])):
            print(f"  {a} <-> {b}: r={r}")
    else:
        print("  없음")
    print()

print("=== 코리도/도심거리 vs 기존 유사 피처 교차 상관 ===")
cross_pairs = [
    ("rn_maxspd", "MAX_SPD"), ("rn_lanes", "LANES"), ("rn_len", "LENGTH"),
    ("dist_cbd", "node_complexity"), ("dist_cbd", "traffic_volume_aadt"),
]
for a, b in cross_pairs:
    r = df[a].corr(df[b])
    print(f"  {a} <-> {b}: r={r:.3f}")

=== 도로기본(수치, 9개) — |r|>=0.8 쌍 ===
  없음



=== 위상·파생(7차, 22개) — |r|>=0.8 쌍 ===
  f_node_outgoing_max_spd_mean <-> t_node_incoming_max_spd_mean: r=0.856
  f_node_outgoing_lanes_mean <-> t_node_incoming_lanes_mean: r=0.84
  t_node_incoming_lanes_mean <-> max_spd_x_lanes: r=0.829
  t_node_incoming_lanes_mean <-> t_node_outgoing_lanes_mean: r=0.826
  f_node_incoming_lanes_mean <-> f_node_outgoing_lanes_mean: r=0.82
  f_node_outgoing_lanes_mean <-> max_spd_x_lanes: r=0.817
  t_node_incoming_max_spd_mean <-> t_node_outgoing_max_spd_mean: r=0.816
  f_node_incoming_max_spd_mean <-> f_node_outgoing_max_spd_mean: r=0.802



=== 생활인구(6차, 2개) — |r|>=0.8 쌍 ===
  없음



=== 강수·첨두파생(7차, 수치 7개) — |r|>=0.8 쌍 ===
  is_rain <-> log_precip: r=0.959
  log_precip <-> rain_x_max_spd: r=0.949
  is_rain <-> rain_x_max_spd: r=0.91



=== ViewT(9차, 6개) — |r|>=0.8 쌍 ===
  congest_freq <-> congest_time: r=0.948
  annual_avg_speed <-> evening_peak_speed: r=0.9
  annual_avg_speed <-> avg_speed_hourly: r=0.871



=== 코리도/도심거리(12차, 5개) — |r|>=0.8 쌍 ===
  rn_links <-> rn_len: r=0.807

=== 코리도/도심거리 vs 기존 유사 피처 교차 상관 ===


  rn_maxspd <-> MAX_SPD: r=0.821


  rn_lanes <-> LANES: r=0.822


  rn_len <-> LENGTH: r=0.322


  dist_cbd <-> node_complexity: r=0.004
  dist_cbd <-> traffic_volume_aadt: r=-0.117


## 9. 공정한 평가 — Section/Link GroupKFold (공간 일반화, 8차~15차)

Model B가 진짜 겨루는 문제는 "한 번도 안 본 도로"를 맞히는 **공간 일반화**(section GroupKFold, T-data 기준 **R²=0.641**)입니다. `LINK_ID` 기준 GroupKFold를 쓰면 폴드마다 특정 링크들이 통째로 test로 빠져서, 그 링크의 어떤 시간대 행도 학습에 들어가지 않습니다.

**중요한 예외 처리**: 이웃속도 계열 피처는 폴드마다 다시 계산합니다(3절·4절 재사용, train 링크만 이웃으로 인정).

**폴드마다 모델을 2개씩 학습합니다(B/C 분리모델)**.

**8차~14차는 재학습하지 않습니다** — `random_state=42`로 고정돼 있어 재실행해도 지난 값과 같으므로, 지난 실행에서 얻은 3-fold 집계값을 그대로 재사용하고 **15차(13차와 동일 피처, LightGBM→XGBoost 교체)만 새로 학습**합니다(폴드 3개 x 모델 2개 = 6번). 3-fold로 진행합니다.

In [12]:
from sklearn.model_selection import GroupKFold
import xgboost as xgb

df["tm_key"] = df["기준일"].astype(str) + "_" + df["시간코드"].astype(int).astype(str)

N_FOLDS = 3
groups = df["LINK_ID"].values
gkf = GroupKFold(n_splits=N_FOLDS)


def build_neighbor_feature_for_allowed(allowed_links):
    """1홉만 — allowed_links(그 폴드의 train 링크)만 이웃으로 인정해 인접 실측 속도 재계산."""
    rows_i, cols_i = [], []
    for l in link_ids_sorted:
        i = link_idx[l]
        for n in link_neighbors.get(l, ()):
            if n in allowed_links:
                rows_i.append(i)
                cols_i.append(link_idx[n])
    A_fold = csr_matrix((np.ones(len(rows_i), dtype="float32"), (rows_i, cols_i)), shape=(n_links, n_links))
    num = A_fold @ filled
    den = A_fold @ valid
    with np.errstate(invalid="ignore", divide="ignore"):
        nm = num / den
    nm[den == 0] = np.nan
    return (
        pd.DataFrame(nm, index=link_ids_sorted, columns=tm_columns)
        .reset_index().rename(columns={"index": "LINK_ID"})
        .melt(id_vars="LINK_ID", var_name="tm_key", value_name="fold_neighbor_speed")
    )


def build_neighbor_feature_2hop_only_for_allowed(allowed_links):
    """2홉전용만(가중치 1.0, 블렌딩 아닌 순수평균) — 폴백용."""
    rows_i, cols_i = [], []
    for l in link_ids_sorted:
        i = link_idx[l]
        for n in link_neighbors_2hop.get(l, ()):
            if n in allowed_links:
                rows_i.append(i)
                cols_i.append(link_idx[n])
    A_fold = csr_matrix((np.ones(len(rows_i), dtype="float32"), (rows_i, cols_i)), shape=(n_links, n_links))
    num = A_fold @ filled
    den = A_fold @ valid
    with np.errstate(invalid="ignore", divide="ignore"):
        nm = num / den
    nm[den == 0] = np.nan
    return (
        pd.DataFrame(nm, index=link_ids_sorted, columns=tm_columns)
        .reset_index().rename(columns={"index": "LINK_ID"})
        .melt(id_vars="LINK_ID", var_name="tm_key", value_name="fold_neighbor_speed")
    )


def build_neighbor_feature_fallback_for_allowed(allowed_links):
    """1홉 우선 + 2홉 폴백(블렌딩 아니라 대체)."""
    primary = build_neighbor_feature_for_allowed(allowed_links).rename(columns={"fold_neighbor_speed": "primary"})
    fallback = build_neighbor_feature_2hop_only_for_allowed(allowed_links).rename(columns={"fold_neighbor_speed": "fallback"})
    merged = primary.merge(fallback, on=["LINK_ID", "tm_key"], how="outer")
    merged["fold_neighbor_speed"] = merged["primary"].fillna(merged["fallback"])
    return merged[["LINK_ID", "tm_key", "fold_neighbor_speed"]]


def make_and_fit(model_type, X_train, y_train, cat_cols, sample_weight=None):
    if model_type == "xgb":
        model = xgb.XGBRegressor(objective="reg:squarederror", n_estimators=300, learning_rate=0.05,
                                 max_depth=8, tree_method="hist", enable_categorical=True,
                                 random_state=RANDOM_STATE, verbosity=0)
        model.fit(X_train, y_train, sample_weight=sample_weight)
    else:
        model = lgb.LGBMRegressor(objective="regression", n_estimators=300, learning_rate=0.05,
                                  num_leaves=63, random_state=RANDOM_STATE, verbosity=-1)
        model.fit(X_train, y_train, categorical_feature=cat_cols + ["dow"], sample_weight=sample_weight)
    return model


# 8차~15차는 random_state=42로 고정돼 있어 재실행해도 지난 실행과 완전히 같은 값이 나옵니다.
# 이번 실행에서 실제로 학습하는 건 16차(13차 + is_cross_t + traffic_volume_ratio + sample_weight, XGBoost)뿐입니다.
CONFIGS = [
    ("16차", FEATURE_COLS_16CHA, "nearest_obs_neighbor_speed_fallback", build_neighbor_feature_fallback_for_allowed,
     CATEGORICAL_COLS_16CHA, "xgb", "sample_count_5min"),
]


def evaluate(y_true, pred, name):
    return {"model": name,
            "rmse": mean_squared_error(y_true, pred) ** 0.5,
            "mae": mean_absolute_error(y_true, pred),
            "r2": r2_score(y_true, pred)}


def eval_bucket(y_true, pred, mask, name):
    if mask.sum() < 30:
        return {"model": name, "n": int(mask.sum()), "rmse": float("nan"), "mae": float("nan"), "r2": float("nan")}
    return {"model": name, "n": int(mask.sum()),
            "rmse": mean_squared_error(y_true[mask], pred[mask]) ** 0.5,
            "mae": mean_absolute_error(y_true[mask], pred[mask]),
            "r2": r2_score(y_true[mask], pred[mask])}


fold_rows = []
fold_tier_rows = []
last_models = {}  # config -> 마지막 폴드의 이웃속도 모델(피처중요도용)

for config_name, feature_cols, neighbor_col, neighbor_builder, cat_cols, model_type, weight_col in CONFIGS:
    static_cols = [c for c in feature_cols if c != neighbor_col]

    for fold_i, (train_idx, test_idx) in enumerate(gkf.split(df, groups=groups)):
        train_links = set(df["LINK_ID"].iloc[train_idx].unique())
        test_links = set(df["LINK_ID"].iloc[test_idx].unique())
        assert len(train_links & test_links) == 0, "링크가 train/test에 겹치면 안 됨"

        nm_long = neighbor_builder(train_links)
        fold_neighbor = df[["LINK_ID", "tm_key"]].merge(nm_long, on=["LINK_ID", "tm_key"], how="left")["fold_neighbor_speed"]

        Xf = df[feature_cols].copy()
        Xf[neighbor_col] = fold_neighbor.to_numpy(dtype="float32")
        yf = df[TARGET]

        X_train, X_test = Xf.iloc[train_idx], Xf.iloc[test_idx]
        y_train, y_test = yf.iloc[train_idx], yf.iloc[test_idx]
        w_train = df[weight_col].iloc[train_idx].to_numpy(dtype="float32") if weight_col else None

        model_nb = make_and_fit(model_type, X_train, y_train, cat_cols, sample_weight=w_train)
        pred_nb = model_nb.predict(X_test)

        model_stat = make_and_fit(model_type, X_train[static_cols], y_train, cat_cols, sample_weight=w_train)
        pred_stat = model_stat.predict(X_test[static_cols])

        has_nb_test = X_test[neighbor_col].notna().to_numpy()
        pred_split = np.where(has_nb_test, pred_nb, pred_stat)
        y_test_arr = y_test.to_numpy()

        row_single = evaluate(y_test_arr, pred_nb, f"{config_name}_fold{fold_i}_단일모델")
        row_split = evaluate(y_test_arr, pred_split, f"{config_name}_fold{fold_i}_분리모델")
        for row, variant in ((row_single, "단일모델"), (row_split, "분리모델")):
            row["config"] = config_name
            row["variant"] = variant
            row["fold"] = fold_i
            row["n_train_links"] = len(train_links)
            row["n_test_links"] = len(test_links)
        fold_rows.append(row_single)
        fold_rows.append(row_split)

        fold_tier_rows.append({**eval_bucket(y_test_arr, pred_nb, has_nb_test, f"{config_name}_단일모델_B"),
                               "config": config_name, "variant": "단일모델", "fold": fold_i, "tier": "B"})
        fold_tier_rows.append({**eval_bucket(y_test_arr, pred_nb, ~has_nb_test, f"{config_name}_단일모델_C"),
                               "config": config_name, "variant": "단일모델", "fold": fold_i, "tier": "C"})
        fold_tier_rows.append({**eval_bucket(y_test_arr, pred_split, has_nb_test, f"{config_name}_분리모델_B"),
                               "config": config_name, "variant": "분리모델", "fold": fold_i, "tier": "B"})
        fold_tier_rows.append({**eval_bucket(y_test_arr, pred_split, ~has_nb_test, f"{config_name}_분리모델_C"),
                               "config": config_name, "variant": "분리모델", "fold": fold_i, "tier": "C"})

        print(f"[{config_name}] fold{fold_i}: train_links={len(train_links):,} test_links={len(test_links):,} "
              f"이웃있음(B) 비율={has_nb_test.mean():.1%}")
        print(f"  단일모델 R2={row_single['r2']:.3f} MAE={row_single['mae']:.3f}"
              f" | 분리모델 R2={row_split['r2']:.3f} MAE={row_split['mae']:.3f}")

        last_models[config_name] = model_nb

        del nm_long, fold_neighbor, Xf, X_train, X_test, y_train, y_test, y_test_arr, pred_nb, pred_stat, pred_split
        gc.collect()

# 8차~15차 지난 실행 결과(재실행 없이 3-fold 집계값 그대로 재사용, 위 설명 참고)
PRIOR_RESULTS = {
    "8차": {
        "분리모델_agg": {"rmse": 6.609346, "mae": 4.606719, "r2": 0.708939},
        "tier": {
            ("단일모델", "B"): {"n": 4748189.667, "rmse": 6.509, "mae": 4.518, "r2": 0.722},
            ("단일모델", "C"): {"n": 283255.333, "rmse": 8.241, "mae": 6.157, "r2": 0.324},
            ("분리모델", "B"): {"n": 4748189.667, "rmse": 6.509, "mae": 4.518, "r2": 0.722},
            ("분리모델", "C"): {"n": 283255.333, "rmse": 8.102, "mae": 6.099, "r2": 0.346},
        },
    },
    "9차": {
        "분리모델_agg": {"rmse": 6.388552, "mae": 4.446168, "r2": 0.728055},
        "tier": {
            ("단일모델", "B"): {"n": 4748189.667, "rmse": 6.288, "mae": 4.357, "r2": 0.741},
            ("단일모델", "C"): {"n": 283255.333, "rmse": 7.985, "mae": 6.000, "r2": 0.364},
            ("분리모델", "B"): {"n": 4748189.667, "rmse": 6.288, "mae": 4.357, "r2": 0.741},
            ("분리모델", "C"): {"n": 283255.333, "rmse": 7.887, "mae": 5.942, "r2": 0.380},
        },
    },
    "10차": {
        "분리모델_agg": {"rmse": 6.368077, "mae": 4.467727, "r2": 0.729799},
        "tier": {
            ("단일모델", "B"): {"n": 5006884.667, "rmse": 6.360, "mae": 4.460, "r2": 0.731},
            ("단일모델", "C"): {"n": 24560.333, "rmse": 8.167, "mae": 6.096, "r2": 0.203},
            ("분리모델", "B"): {"n": 5006884.667, "rmse": 6.360, "mae": 4.460, "r2": 0.731},
            ("분리모델", "C"): {"n": 24560.333, "rmse": 7.884, "mae": 6.008, "r2": 0.248},
        },
    },
    "11차": {
        "분리모델_agg": {"rmse": 6.368507, "mae": 4.427996, "r2": 0.729759},
        "tier": {
            ("단일모델", "B"): {"n": 5006884.667, "rmse": 6.360, "mae": 4.420, "r2": 0.731},
            ("단일모델", "C"): {"n": 24560.333, "rmse": 8.122, "mae": 6.094, "r2": 0.211},
            ("분리모델", "B"): {"n": 5006884.667, "rmse": 6.360, "mae": 4.420, "r2": 0.731},
            ("분리모델", "C"): {"n": 24560.333, "rmse": 7.884, "mae": 6.008, "r2": 0.248},
        },
    },
    "12차": {
        "분리모델_agg": {"rmse": 6.288000, "mae": 4.380929, "r2": 0.736553},
        "tier": {
            ("단일모델", "B"): {"n": 5006884.667, "rmse": 6.280, "mae": 4.374, "r2": 0.738},
            ("단일모델", "C"): {"n": 24560.333, "rmse": 7.885, "mae": 5.943, "r2": 0.255},
            ("분리모델", "B"): {"n": 5006884.667, "rmse": 6.280, "mae": 4.374, "r2": 0.738},
            ("분리모델", "C"): {"n": 24560.333, "rmse": 7.784, "mae": 5.879, "r2": 0.272},
        },
    },
    "13차": {
        "분리모델_agg": {"rmse": 6.282637, "mae": 4.377409, "r2": 0.736997},
        "tier": {
            ("단일모델", "B"): {"n": 5006884.667, "rmse": 6.275, "mae": 4.370, "r2": 0.738},
            ("단일모델", "C"): {"n": 24560.333, "rmse": 7.836, "mae": 5.914, "r2": 0.264},
            ("분리모델", "B"): {"n": 5006884.667, "rmse": 6.275, "mae": 4.370, "r2": 0.738},
            ("분리모델", "C"): {"n": 24560.333, "rmse": 7.686, "mae": 5.798, "r2": 0.288},
        },
    },
    "14차": {
        "분리모델_agg": {"rmse": 6.287636, "mae": 4.384043, "r2": 0.736582},
        "tier": {
            ("단일모델", "B"): {"n": 5006884.667, "rmse": 6.279, "mae": 4.377, "r2": 0.738},
            ("단일모델", "C"): {"n": 24560.333, "rmse": 7.943, "mae": 5.929, "r2": 0.245},
            ("분리모델", "B"): {"n": 5006884.667, "rmse": 6.279, "mae": 4.377, "r2": 0.738},
            ("분리모델", "C"): {"n": 24560.333, "rmse": 7.802, "mae": 5.919, "r2": 0.268},
        },
    },
    "15차": {
        "분리모델_agg": {"rmse": 6.245877, "mae": 4.324281, "r2": 0.740064},
        "tier": {
            ("단일모델", "B"): {"n": 5006884.667, "rmse": 6.239, "mae": 4.317, "r2": 0.741},
            ("단일모델", "C"): {"n": 24560.333, "rmse": 7.987, "mae": 5.998, "r2": 0.235},
            ("분리모델", "B"): {"n": 5006884.667, "rmse": 6.239, "mae": 4.317, "r2": 0.741},
            ("분리모델", "C"): {"n": 24560.333, "rmse": 7.543, "mae": 5.723, "r2": 0.315},
        },
    },
}

for config_name, prior in PRIOR_RESULTS.items():
    fold_rows.append({"model": f"{config_name}_prior_분리모델", "config": config_name,
                       "variant": "분리모델", **prior["분리모델_agg"]})
    for (variant, tier), vals in prior["tier"].items():
        fold_tier_rows.append({"model": f"{config_name}_{variant}_{tier}", "config": config_name,
                               "variant": variant, "tier": tier, **vals})

group_results = pd.DataFrame(fold_rows)
group_tier_results = pd.DataFrame(fold_tier_rows)
group_results


[16차] fold0: train_links=29,973 test_links=14,987 이웃있음(B) 비율=99.5%
  단일모델 R2=0.738 MAE=4.326 | 분리모델 R2=0.738 MAE=4.326


[16차] fold1: train_links=29,974 test_links=14,986 이웃있음(B) 비율=99.5%
  단일모델 R2=0.736 MAE=4.327 | 분리모델 R2=0.736 MAE=4.326


[16차] fold2: train_links=29,973 test_links=14,987 이웃있음(B) 비율=99.5%
  단일모델 R2=0.747 MAE=4.318 | 분리모델 R2=0.747 MAE=4.316


,model,rmse,mae,r2,config,variant,fold,n_train_links,n_test_links
0,16차_fold0_단일모델,6.249600,4.326208,0.738057,16차,단일모델,0.0,29973.0,14987.0
1,16차_fold0_분리모델,6.249385,4.326313,0.738075,16차,분리모델,0.0,29973.0,14987.0
2,16차_fold1_단일모델,6.305968,4.327332,0.735620,16차,단일모델,1.0,29974.0,14986.0
3,16차_fold1_분리모델,6.304523,4.326416,0.735741,16차,분리모델,1.0,29974.0,14986.0
4,16차_fold2_단일모델,6.179542,4.317633,0.746739,16차,단일모델,2.0,29973.0,14987.0
5,16차_fold2_분리모델,6.175692,4.316160,0.747054,16차,분리모델,2.0,29973.0,14987.0
6,8차_prior_분리모델,6.609346,4.606719,0.708939,8차,분리모델,NaN,NaN,NaN
7,9차_prior_분리모델,6.388552,4.446168,0.728055,9차,분리모델,NaN,NaN,NaN
8,10차_prior_분리모델,6.368077,4.467727,0.729799,10차,분리모델,NaN,NaN,NaN
9,11차_prior_분리모델,6.368507,4.427996,0.729759,11차,분리모델,NaN,NaN,NaN


In [13]:
# 계층(B/C)별 3-fold 평균
tier_summary = group_tier_results.groupby(["config", "variant", "tier"])[["n", "rmse", "mae", "r2"]].mean()
print("=== 계층별 3-fold 평균 (공간 일반화, 8차~15차) ===")
print(tier_summary.round(3).to_string())
print()
for config_name in ["8차", "9차", "10차", "11차", "12차", "13차", "14차", "15차"]:
    b_single = tier_summary.loc[(config_name, "단일모델", "B")]
    c_single = tier_summary.loc[(config_name, "단일모델", "C")]
    b_split = tier_summary.loc[(config_name, "분리모델", "B")]
    c_split = tier_summary.loc[(config_name, "분리모델", "C")]
    print(f"[{config_name}] B 계층 n={b_single['n']:,.0f} R2={b_single['r2']:.3f} | "
          f"C 계층 n={c_single['n']:,.0f} 단일 R2={c_single['r2']:.3f} -> 분리 R2={c_split['r2']:.3f} "
          f"(Δ{c_split['r2']-c_single['r2']:+.3f})")


=== 계층별 3-fold 평균 (공간 일반화, 8차~15차) ===
                               n   rmse    mae     r2
config variant tier                                  
10차    단일모델    B     5006884.667  6.360  4.460  0.731
               C       24560.333  8.167  6.096  0.203
       분리모델    B     5006884.667  6.360  4.460  0.731
               C       24560.333  7.884  6.008  0.248
11차    단일모델    B     5006884.667  6.360  4.420  0.731
               C       24560.333  8.122  6.094  0.211
       분리모델    B     5006884.667  6.360  4.420  0.731
               C       24560.333  7.884  6.008  0.248
12차    단일모델    B     5006884.667  6.280  4.374  0.738
               C       24560.333  7.885  5.943  0.255
       분리모델    B     5006884.667  6.280  4.374  0.738
               C       24560.333  7.784  5.879  0.272
13차    단일모델    B     5006884.667  6.275  4.370  0.738
               C       24560.333  7.836  5.914  0.264
       분리모델    B     5006884.667  6.275  4.370  0.738
               C       24560.333  7.686  5.

In [14]:
split_fold_results = group_results[group_results["variant"] == "분리모델"]
group_summary_by_config = split_fold_results.groupby("config")[["rmse", "mae", "r2"]].mean()

print("=== 3-fold 평균 (공간 일반화, B/C 분리모델, 정직한 비교) ===")
print(group_summary_by_config.round(3).to_string())
print()
print("=== 벤치마크 비교표 ===")
compare = pd.DataFrame([
    {"model": "Model B 공간일반화 (T-data, 팀 공식 벤치마크)", "rmse": 7.71, "mae": 5.61, "r2": 0.641},
    {"model": "Model B 공간일반화 (5분 데이터, src/11 재실행, 우리 피처 없음)", "rmse": 8.104321, "mae": 5.947932, "r2": 0.562342},
    {"model": "Model B 시간일반화 (T-data, 참고용)", "rmse": 6.43, "mae": 4.72, "r2": 0.740},
    {"model": "8차(위상+강수+인구, B/C 분리모델)", **group_summary_by_config.loc["8차"].to_dict()},
    {"model": "9차(8차 + ViewT 평균속도/혼잡, B/C 분리모델)", **group_summary_by_config.loc["9차"].to_dict()},
    {"model": "10차(9차 + 2홉 블렌딩, B/C 분리모델)", **group_summary_by_config.loc["10차"].to_dict()},
    {"model": "11차(9차 + 2홉 폴백, B/C 분리모델)", **group_summary_by_config.loc["11차"].to_dict()},
    {"model": "12차(11차 + 코리도/도심거리 + 위상·고도 중복제거, B/C 분리모델)", **group_summary_by_config.loc["12차"].to_dict()},
    {"model": "13차(12차 + gain=0 13개 제거, B/C 분리모델) *현재 최선*", **group_summary_by_config.loc["13차"].to_dict()},
    {"model": "14차(13차 + 추가정리9개+log_precip복원+회전제한/교차로4개)", **group_summary_by_config.loc["14차"].to_dict()},
    {"model": "15차(13차와 동일 피처, LightGBM->XGBoost)", **group_summary_by_config.loc["15차"].to_dict()},
    {"model": "16차(15차 + is_cross_t + traffic_volume_ratio + sample_weight) *현재 최선*", **group_summary_by_config.loc["16차"].to_dict()},
])
print(compare[["model", "rmse", "mae", "r2"]].to_string(index=False))

=== 3-fold 평균 (공간 일반화, B/C 분리모델, 정직한 비교) ===
         rmse    mae     r2
config                     
10차     6.368  4.468  0.730
11차     6.369  4.428  0.730
12차     6.288  4.381  0.737
13차     6.283  4.377  0.737
14차     6.288  4.384  0.737
15차     6.246  4.324  0.740
16차     6.243  4.323  0.740
8차      6.609  4.607  0.709
9차      6.389  4.446  0.728

=== 벤치마크 비교표 ===
                                                               model     rmse      mae       r2
                                   Model B 공간일반화 (T-data, 팀 공식 벤치마크) 7.710000 5.610000 0.641000
                        Model B 공간일반화 (5분 데이터, src/11 재실행, 우리 피처 없음) 8.104321 5.947932 0.562342
                                         Model B 시간일반화 (T-data, 참고용) 6.430000 4.720000 0.740000
                                              8차(위상+강수+인구, B/C 분리모델) 6.609346 4.606719 0.708939
                                    9차(8차 + ViewT 평균속도/혼잡, B/C 분리모델) 6.388552 4.446168 0.728055
                                          10차(9차 + 2홉

## 10. 피처 중요도 (16차, GroupKFold 마지막 폴드 이웃속도모델 기준)

XGBoost의 `get_score(importance_type="gain")`으로 gain 기준 중요도를 봅니다. LightGBM과 API가 달라(사용 안 된 피처는 딕셔너리에 아예 없음) 0으로 채워 전체를 다 보여줍니다.

In [15]:
model_16cha = last_models["16차"]
booster = model_16cha.get_booster()
score = booster.get_score(importance_type="gain")
importance = pd.DataFrame({"feature": FEATURE_COLS_16CHA})
importance["gain"] = importance["feature"].map(score).fillna(0.0)
importance = importance.sort_values("gain", ascending=False)
importance["gain_share"] = importance["gain"] / importance["gain"].sum()

NEIGHBOR_COL_NAMES = ["nearest_obs_neighbor_speed", "nearest_obs_neighbor_speed_2hop", "nearest_obs_neighbor_speed_fallback"]
NEW_COLS_ALL = (DERIVED_COLS + POPULATION_COLS + TOPO_COLS + RAIN_PEAK_NUMERIC_COLS + RAIN_PEAK_CATEGORICAL_COLS
                + VIEWT_COLS + NEIGHBOR_COL_NAMES + CORRIDOR_COLS + TURN_COLS)
importance["group"] = np.select(
    [importance["feature"].isin(NEIGHBOR_COL_NAMES),
     importance["feature"].isin(CORRIDOR_COLS),
     importance["feature"].isin(TURN_COLS),
     importance["feature"].isin(VIEWT_COLS),
     importance["feature"].isin(TOPO_COLS),
     importance["feature"].isin(RAIN_PEAK_NUMERIC_COLS + RAIN_PEAK_CATEGORICAL_COLS),
     importance["feature"].isin(POPULATION_COLS),
     importance["feature"].isin(EXTERNAL_CONTEXT_COLS + SPARSE_NUMERIC_COLS)],
    ["이웃속도(폴백)", "코리도/도심거리(12차)", "회전제한/교차로유형(14차)", "ViewT(9차)", "위상·파생(7차)",
     "강수·첨두파생(7차)", "생활인구(6차)", "외부컨텍스트"],
    default="도로기본/지형/시간",
)
importance["is_new"] = importance["feature"].isin(NEW_COLS_ALL)

pd.set_option("display.max_rows", None)
print(f"=== 전체 피처 gain 목록 (16차 XGBoost, {len(FEATURE_COLS_16CHA)}개 전부) ===")
print(importance[["feature", "group", "gain_share"]].round(4).to_string(index=False))
print()
print("=== 그룹별 gain 합계 ===")
print(importance.groupby("group")["gain_share"].sum().sort_values(ascending=False).round(4).to_string())
pd.reset_option("display.max_rows")


=== 전체 피처 gain 목록 (16차 XGBoost, 61개 전부) ===
                            feature           group  gain_share
nearest_obs_neighbor_speed_fallback        이웃속도(폴백)      0.1654
                            MAX_SPD      도로기본/지형/시간      0.1363
                         is_cross_t 회전제한/교차로유형(14차)      0.1085
                   avg_speed_hourly       ViewT(9차)      0.0711
                          ROAD_TYPE      도로기본/지형/시간      0.0327
                          rn_maxspd   코리도/도심거리(12차)      0.0289
                    max_spd_x_lanes       위상·파생(7차)      0.0225
             f_node_node_out_degree       위상·파생(7차)      0.0212
                             rn_len   코리도/도심거리(12차)      0.0204
             downstream_speed_delta       위상·파생(7차)      0.0190
                   annual_avg_speed       ViewT(9차)      0.0170
              t_node_node_in_degree       위상·파생(7차)      0.0159
             t_node_node_out_degree       위상·파생(7차)      0.0154
                           REST_VEH      도로기본/지형/시간      0.0

## 11. 해석 및 한계

**해석 기준**: 8차~16차 모두 `group_summary_by_config`(공간 일반화, B/C 분리모델)를 Model B 공간일반화 두 벤치마크(T-data 0.641, 5분 데이터+팀 피처만 0.562342)와 비교하는 게 정직한 비교입니다.

**요약 흐름**: 8차(0.709) → 9차(0.728, View-T) → 10/11차(0.730, 2홉 확장) → 12차(0.737, 코리도/도심거리+위상정리) → 13차(0.737, 59개로 경량화) → 14차(0.737, 소폭 하락, 채택 안 함) → 15차(13차와 동일 피처, LightGBM→XGBoost, 0.740) → **16차(15차 + is_cross_t + traffic_volume_ratio + sample_weight, 0.740, 사실상 정체 — 이 계열 개선은 한계에 도달)**.

**13차 대비 15차 효과**: 피처는 완전히 동일하고 알고리즘만 LightGBM→XGBoost로 바꿨습니다. `notebooks/01_speed_model_baseline.ipynb`의 무작위분할 베이스라인(원본 피처만)에서도 XGBoost가 LightGBM을 앞섰는데(R² 0.610→0.632), 이 우위가 GroupKFold·B/C분리모델 조건에서도 재현됐습니다 — 이후 XGBoost를 기본 알고리즘으로 채택했습니다.

**계층별 결과(9절)가 핵심**입니다 — B/C 분리모델을 쓰면 C 계층(이웃없음)의 R²가 단일모델 대비 얼마나 회복되는지가, "이웃속도 피처를 그대로 쓰면 정작 정보가 가장 부족한 링크에서 해를 끼친다"는 13번 노트북의 발견을 재현하는지 보여줍니다.

**배포 단계에서 발견한 것(중요)**: 이 노트북의 GroupKFold 평가는 이웃속도 피처를 "그 폴드 train 링크가 정확히 같은 날짜·시간에 실측한 값"으로 계산합니다 — 그런데 배포 시점엔 라이브 피드가 없어 미래 날짜의 실측값을 구할 수 없어 이 피처를 그대로 쓸 수 없습니다. `src/models/02_build_model_bc.py`는 이웃속도를 "그 이웃의 (시간코드, 주중/주말) 버킷 평균"(01번 스크립트의 Model A와 동일 방식)으로 다시 정의해 학습·서빙 정의를 통일했고, 같은 GroupKFold 틀로 재검증한 결과 R²=0.7226(이 노트북의 0.740 대비 -0.018)이 나왔습니다 — "그 순간의 실제 값"이 담고 있던 정보(그 순간 공유되는 날씨·혼잡) 일부가 버킷 평균으로는 복원되지 않기 때문입니다. 이 노트북의 0.740은 피처 조합·알고리즘 선택의 근거로는 유효하지만, 실제 배포 성능을 대표하는 숫자는 아닙니다.

**남은 가정/한계**
- 추정교통량 시간대 매칭은 순서 기반 가정이고, 2024년 평일 기준이라 2026년 7월 관측과 시점·요일 구성이 다릅니다.
- 스쿨존 200m 버퍼는 직선거리 근사이고, 과속방지턱 원본 도형은 `on_invalid="fix"`로 보정해서 읽었습니다.
- 생활인구(5절) 결측률은 0.5% 이하입니다.
- 도로망 위상·파생(1절)은 `notebooks/04_tier_split_evaluation.ipynb`가 **공간 블록분할** 기준으로 먼저 검증한 값이라, 이 노트북의 GroupKFold 결과가 그 규모를 그대로 재현하지 않을 수 있습니다.
- 회전제한/교차로유형(7-1절)은 `MOCT_NODE.shp`/`TURNINFO.dbf`가 **전국** 기준이라 서울 경계 인접 노드에서는 실제보다 차수·회전제한이 과소집계될 수 있습니다.
- XGBoost(15/16차)는 하이퍼파라미터를 LightGBM과 유사한 기본값(`n_estimators=300, learning_rate=0.05, max_depth=8`)으로만 맞췄고 별도 튜닝은 안 했습니다 — 시간 대비 기대 이득이 작다고 판단해 보류했습니다(검토 근거는 대화 기록 참고).

**배포 현황**: Model A(관측 링크 조회표) + Model B/C(미관측 링크 XGBoost, 이 노트북의 16차 피처셋 기반)가 `src/models/01_build_model_a.py`·`02_build_model_bc.py`로 정식 구현·학습·저장되었습니다(`src/models/artifacts/model_nb.pkl`, `model_stat.pkl`, `data/processed/model_a_lookup.parquet`, `neighbor_speed_lookup.parquet`). 전 링크 예측은 `src/models/03·04`가 재학습 없이 이 아티팩트를 불러와 수행합니다.